# Griffschrift: Kopf-Erkenner, dritter Lernlauf (YOLO)

Neu gegenüber dem zweiten Lauf: alle Fuchs-Seiten mit bekannten Griffen (außer den 30 Prüfseiten) statt 19, dazu echte Fotos (Muchitsch-Heft, Rosenzopf Heft 2), vorbeschriftet mit dem bisherigen und dem zweiten Erkenner; unsichere Stellen werden weiß abgedeckt. Gelernt wird von vorn (vom zweiten Lauf gibt es nur das ONNX), die Gewichte werden laufend auf Google Drive gespeichert (griffschrift_modell/lauf3), damit ein Abbruch nichts kostet.

Geprüft wird an den 30 Fuchs-Prüfseiten und an Blättern, die nicht gelernt werden: die Rosenzopf-Fotos vom 3.11., handgeschriebene und eingesandte PDFs. Alle Ergebnisse landen in griffschrift_modell/ergebnis3.txt.

Vorher: Laufzeit → Laufzeittyp ändern → T4-GPU. Dann Laufzeit → Alle ausführen. Dauer etwa 3–4 Stunden.

In [ ]:
#@title 1. Einrichten, Drive verbinden (ca. 5 Minuten)
from google.colab import drive
drive.mount('/content/drive')
%cd /content
!rm -rf /content/Harmonikabau && git clone -q --depth 1 https://github.com/jpascher/Harmonikabau.git
!pip -q install ultralytics opencv-python-headless onnxruntime scikit-learn==1.9.1 mido pymupdf
!npm -s i playwright@1 > /dev/null
!npx -y playwright install --with-deps chromium > /dev/null
import torch, os
print('Grafikkarte:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'KEINE – bitte T4-GPU einstellen')
M = '/content/drive/MyDrive/griffschrift_modell'; os.makedirs(M, exist_ok=True)
def log(*a):
    print(*a); open(M + '/ergebnis3.txt', 'a').write(' '.join(map(str, a)) + '\n')

In [ ]:
%%writefile /content/Harmonikabau/griffschrift-wandler/omr-space/griffschrift-leser/yolo_foto_seiten.py
# Echte Lernseiten für den Kopf-Erkenner (YOLO) aus Fotos und Scans ohne bekannte Lösung (dritter Lernlauf).
# Vorbeschriftet wird mit zwei Erkennern: Rahmen bekommen die Köpfe, bei denen der bisherige Erkenner (detect.py) sehr sicher ist,
# und die Köpfe, die nur das gelernte Modell (yolo_koepfe.py, falls vorhanden) mit hoher Sicherheit meldet. Wo einer der beiden
# etwas Unsicheres meldet, das der andere nicht bestätigt, wird die Stelle im Bild weiß abgedeckt: so lernt das Modell dort weder
# „Kopf“ noch „kein Kopf“. Vorgesetzte Doppelkreuze (Rosenzopf) kommen nur vom gelernten Modell; liest der bisherige Erkenner an
# derselben Stelle einen Kopf, gilt das Doppelkreuz. Für die Form zählt das Urteil des bisherigen Erkenners, weil das gelernte Modell
# auf Fotos volle Köpfe öfter als hohle liest.
# Ausgabe wie yolo_fuchs_seiten.py (PNG + JSON mit boxes), dazu "foto": true, damit yolo_daten.py die Seite nicht noch einmal
# wie ein Foto verschlechtert. Die Blätter selbst kommen nicht ins Repository.
# Aufruf: [GS_YOLOMODELL=…/griffschrift_koepfe.onnx] python3 yolo_foto_seiten.py AUSGABEORDNER BILD [BILD …]
import sys, os, json
import numpy as np, cv2
HERE = os.path.dirname(os.path.abspath(__file__)); sys.path.insert(0, os.path.join(HERE, '..', 'gsleser'))
import gs2xml as X, detect as D, gsread as G

GR = {0: (1.45, 1.15), 1: (1.45, 1.15), 2: (1.2, 1.2), 3: (1.35, 1.35), 4: (1.35, 1.1), 5: (1.35, 1.1), 6: (1.0, 1.0)}
SICHER = float(os.environ.get('FS_SICHER', '0.9'))      # bisheriger Erkenner: ab hier Rahmen
YSICHER = float(os.environ.get('FS_YSICHER', '0.7'))    # gelerntes Modell: ab hier Rahmen, wo der bisherige nichts meldet
YUNTEN = float(os.environ.get('FS_YUNTEN', '0.3'))      # gelerntes Modell: darunter gilt eine Meldung als nichts


def klasse(h):
    if h['t'] == 'o': return 0 if h['fill'] else 1
    if h['t'] == 'x': return 2 if h['fill'] else 3
    return 4 if h['fill'] else 5


def seite(path):
    g = X.load_gray(path)
    if g is None: return None
    work = '/tmp/_yfo.png'; cv2.imwrite(work, g)
    det = [h for h in D.detect(work, 1) if abs(h['y'] - h['mid']) < 9 * h['d']]
    yk = []
    try:
        import yolo_koepfe as YK
        if YK.vorhanden(): yk = [k for k in YK.koepfe(g, conf=YUNTEN)]
    except Exception as e: print('  Modell nicht nutzbar:', str(e)[:80])
    d0 = float(np.median([h['d'] for h in det])) if det else 18.0
    nah = lambda x1, y1, x2, y2, f=0.6: abs(x1 - x2) < f * d0 and abs(y1 - y2) < f * d0
    vork = [(cx, cy) for cx, cy, w, h, k, sc in yk if k == 6 and sc >= 0.5]
    boxen = []; weiss = []
    for h in det:
        x, y, d = h['x'], h['y'], h['d']
        if any(nah(x, y, vx, vy) for vx, vy in vork): continue        # Doppelkreuz, kein Kopf
        bestaetigt = any(nah(x, y, cx, cy) and k != 6 for cx, cy, w, hh, k, sc in yk)
        if h.get('sc', 0) >= SICHER or (bestaetigt and h.get('sc', 0) >= 0.6):
            k = klasse(h); w, hh = GR[k][0] * d, GR[k][1] * d; boxen.append((k, x - w / 2, y - hh / 2, w, hh))
        else: weiss.append((x, y, d))
    for cx, cy, w, hh, k, sc in yk:
        if any(nah(cx, cy, h['x'], h['y']) for h in det) and k != 6: continue
        if sc >= YSICHER: boxen.append((k, cx - w / 2, cy - hh / 2, w, hh))
        else: weiss.append((cx, cy, d0))
    g2 = g.copy(); hell = int(np.percentile(g, 90))
    for x, y, d in weiss:   # unsichere Stellen mit Papierhelligkeit abdecken, aber nicht über einen beschrifteten Kopf
        x0, y0, x1, y1 = int(x - 0.8 * d), int(y - 0.65 * d), int(x + 0.8 * d), int(y + 0.65 * d)
        if any(bx < x1 and bx + bw > x0 and by < y1 and by + bh > y0 for _, bx, by, bw, bh in boxen): continue
        g2[max(0, y0):y1, max(0, x0):x1] = hell
    return dict(g=g2, boxen=boxen, weiss=len(weiss), alt=len(det), yolo=len(yk), foto=bool(G.FOTO_LINES))


if __name__ == '__main__':
    ziel = sys.argv[1]; os.makedirs(ziel, exist_ok=True)
    for p in sys.argv[2:]:
        k = os.path.splitext(os.path.basename(p))[0].replace(' ', '_')
        try: r = seite(p)
        except Exception as e: print(k, 'Fehler', str(e)[:80]); continue
        if r is None or len(r['boxen']) < 30: print(k, 'ausgelassen', r and len(r['boxen'])); continue
        cv2.imwrite(os.path.join(ziel, f'foto_{k}.png'), r['g'])
        json.dump(dict(foto=True, boxes=[[int(c), round(x, 1), round(y, 1), round(w, 1), round(h, 1)] for c, x, y, w, h in r['boxen']]),
                  open(os.path.join(ziel, f'foto_{k}.json'), 'w'))
        print(k, 'Köpfe', len(r['boxen']), 'abgedeckt', r['weiss'], '(bisher', r['alt'], 'gelernt', r['yolo'], ')', 'Foto' if r['foto'] else 'Scan', flush=True)


In [ ]:
%%writefile /content/Harmonikabau/griffschrift-wandler/omr-space/griffschrift-leser/yolo_daten.py
# Trainingsdaten für den Kopf-Erkenner (YOLO) aus den Systembildern von yolo_erzeuge_wandlerbilder.js:
# mehrere Systeme werden zu einer Seite gestapelt, die Seite wird zufällig wie ein Scan, eine Kopie oder ein Handyfoto verschlechtert
# (Unschärfe, Rauschen, ungleiche Helligkeit, leichte Drehung und Verzerrung, dünnere oder dickere Striche, JPEG) und in Ausschnitte
# von 960 × 960 Bildpunkten geschnitten. Ausgabe im YOLO-Format (images/, labels/, data.yaml).
# Jedes System wird auf einen Linienabstand um 18 Bildpunkte gebracht, wie ihn der Leser (load_gray) für jede Seite herstellt.
# Ein Teil der Seiten wird wie die Fuchs-Blätter aus einer niedrig aufgelösten Vorlage hochgerechnet. Liegen echte Seiten mit
# Rahmen vor (yolo_fuchs_seiten.py, yolo_foto_seiten.py), wird jede vierte Seite (ECHT_JEDE) eine davon, jedesmal anders verschlechtert;
# echte Fotos ("foto": true) nur leicht, weil sie schon wie Fotos aussehen.
# Aufruf: python3 yolo_daten.py ROHORDNER AUSGABEORDNER [SEITEN] [ECHTORDNER]
import sys, os, json, glob, random
import numpy as np, cv2

KLASSEN = ['rund_voll', 'rund_hohl', 'kreuz', 'kreuz_kreis', 'dreieck_voll', 'dreieck_hohl', 'vorkreuz']
T = 960   # Ausschnittgröße


def lade(raw):
    out = []
    for f in sorted(glob.glob(os.path.join(raw, '*.json'))):
        j = json.load(open(f)); im = cv2.imread(f[:-5] + '.png', cv2.IMREAD_GRAYSCALE)
        if im is None or not j.get('boxes'): continue
        s = im.shape[1] / j['w']; ys = j.get('staff') or []
        d = (ys[4] - ys[0]) / 4 * s if len(ys) >= 5 else 8 * s
        out.append((im, [(c, x * s, y * s, w * s, h * s) for c, x, y, w, h in j['boxes']], d))
    return out


def lade_echt(ordner):
    out = []
    for f in sorted(glob.glob(os.path.join(ordner or '', '*.json'))):
        im = cv2.imread(f[:-5] + '.png', cv2.IMREAD_GRAYSCALE); j = json.load(open(f))
        if im is not None and j.get('boxes'): out.append((im, [tuple(b) for b in j['boxes']], bool(j.get('foto'))))
    return out


def echte_seite(echt, rnd):
    im, bx, foto = rnd.choice(echt); k = rnd.uniform(0.88, 1.12)
    im = cv2.resize(im, None, fx=k, fy=k, interpolation=cv2.INTER_AREA if k < 1 else cv2.INTER_LINEAR)
    return im, [(c, x * k, y * k, w * k, h * k) for c, x, y, w, h in bx], foto


def leicht(g, rnd):
    """echte Fotos sind schon unscharf und ungleich hell: nur etwas Rauschen, Helligkeit und JPEG"""
    g = g.astype(np.float32) * rnd.uniform(0.85, 1.05) + rnd.uniform(-15, 10) + np.random.normal(0, rnd.uniform(0, 4), g.shape)
    g = np.clip(g, 0, 255).astype(np.uint8)
    return cv2.imdecode(cv2.imencode('.jpg', g, [cv2.IMWRITE_JPEG_QUALITY, rnd.randint(50, 92)])[1], cv2.IMREAD_GRAYSCALE)


def grob(pg, rnd):
    """wie eine niedrig aufgelöste Vorlage (GIF), die hochgerechnet wurde: Punkte werden Kästchen, Linien grau und unscharf"""
    f = rnd.uniform(2.3, 3.2); H, W = pg.shape
    kl = cv2.resize(pg, (max(1, int(W / f)), max(1, int(H / f))), interpolation=cv2.INTER_AREA)
    if rnd.random() < 0.6: kl = np.where(kl < rnd.randint(110, 170), 0, 255).astype(np.uint8)
    return cv2.resize(kl, (W, H), interpolation=rnd.choice([cv2.INTER_LINEAR, cv2.INTER_CUBIC, cv2.INTER_NEAREST]))


def seite(sys_, rnd):
    """4–8 Systeme untereinander auf weißem Blatt, auf eine gemeinsame Breite gebracht."""
    dz = rnd.uniform(15.5, 21.0); teile = []; boxen = []; y = rnd.randint(60, 250)
    wahl = rnd.sample(sys_, k=min(len(sys_), rnd.randint(4, 8)))
    W = max(int(im.shape[1] * dz / d) for im, _, d in wahl) + 40
    for im, bx, d in wahl:
        k = dz / d * rnd.uniform(0.97, 1.03); im2 = cv2.resize(im, None, fx=k, fy=k, interpolation=cv2.INTER_AREA if k < 1 else cv2.INTER_LINEAR)
        x0 = rnd.randint(20, max(21, W - im2.shape[1] + 20))
        teile.append((im2, x0, y)); boxen += [(c, x0 + x * k, y + yy * k, w * k, h * k) for c, x, yy, w, h in bx]
        y += im2.shape[0] + rnd.randint(10, 120)
    H = y + rnd.randint(60, 250); pg = np.full((H, W + 60), 255, np.uint8)
    for im2, x0, y0 in teile:
        h, w = im2.shape; pg[y0:y0 + h, x0:x0 + w] = np.minimum(pg[y0:y0 + h, x0:x0 + w], im2[:, :min(w, pg.shape[1] - x0)])
    return pg, boxen


def verschlechtern(pg, boxen, rnd):
    art = rnd.choice(['sauber', 'scan', 'kopie', 'foto', 'foto'])
    g = pg.astype(np.float32)
    if art in ('kopie', 'foto'):   # Striche dünner oder dicker (auf Fotos auch deutlich dicker)
        k = np.ones((3, 3) if art == 'foto' and rnd.random() < 0.4 else (2, 2), np.uint8)
        g = cv2.erode(g, k) if rnd.random() < 0.55 else cv2.dilate(g, k)
    if art != 'sauber':
        g = cv2.GaussianBlur(g, (0, 0), rnd.uniform(0.3, 2.2 if art == 'foto' else 0.9))
    H, W = g.shape
    M = np.eye(3, dtype=np.float32)
    if art in ('scan', 'kopie', 'foto'):   # leichte Drehung, bei Fotos auch Verzerrung
        a = rnd.uniform(-2.5, 2.5) if art == 'foto' else rnd.uniform(-1, 1)
        R = cv2.getRotationMatrix2D((W / 2, H / 2), a, 1.0); M = np.vstack([R, [0, 0, 1]]).astype(np.float32)
        if art == 'foto':
            d = 0.03 * min(W, H); src = np.float32([[0, 0], [W, 0], [W, H], [0, H]])
            dst = src + np.float32([[rnd.uniform(-d, d), rnd.uniform(-d, d)] for _ in range(4)])
            M = cv2.getPerspectiveTransform(src, dst) @ M
        g = cv2.warpPerspective(g, M, (W, H), borderValue=255)
    if art == 'foto':   # ungleiche Helligkeit, grauer Hintergrund, Rauschen
        yy, xx = np.mgrid[0:H, 0:W].astype(np.float32)
        grad = 0.55 + 0.4 * (np.cos(xx / W * rnd.uniform(1, 4) + rnd.uniform(0, 6)) * 0.5 + 0.5) * (np.cos(yy / H * rnd.uniform(1, 4)) * 0.5 + 0.5)
        g = g * grad + rnd.uniform(-20, 10)
        g = g + np.random.normal(0, rnd.uniform(3, 12), g.shape)
        g = (g - g.min()) / max(1e-3, g.max() - g.min()) * rnd.uniform(110, 255) + rnd.uniform(0, 70)   # auch flau: graue Köpfe auf grauem Papier
    elif art in ('scan', 'kopie'):
        g = g + np.random.normal(0, rnd.uniform(1, 6), g.shape)
    g = np.clip(g, 0, 255).astype(np.uint8)
    if art != 'sauber':
        q = rnd.randint(35, 90); g = cv2.imdecode(cv2.imencode('.jpg', g, [cv2.IMWRITE_JPEG_QUALITY, q])[1], cv2.IMREAD_GRAYSCALE)
    nb = []
    for c, x, y, w, h in boxen:   # Rahmen mitbewegen (Ecken umrechnen, umschließendes Rechteck)
        pts = np.float32([[x, y], [x + w, y], [x + w, y + h], [x, y + h]]).reshape(-1, 1, 2)
        p2 = cv2.perspectiveTransform(pts, M).reshape(-1, 2)
        x0, y0 = p2.min(0); x1, y1 = p2.max(0); nb.append((c, x0, y0, x1 - x0, y1 - y0))
    return g, nb


def ausschnitte(g, boxen, rnd, n):
    H, W = g.shape; out = []
    for _ in range(n):
        x0 = rnd.randint(0, max(0, W - T)); y0 = rnd.randint(0, max(0, H - T))
        crop = np.full((T, T), 255, np.uint8); c_ = g[y0:y0 + T, x0:x0 + T]; crop[:c_.shape[0], :c_.shape[1]] = c_
        lab = []
        for c, x, y, w, h in boxen:
            cx, cy = x + w / 2 - x0, y + h / 2 - y0
            if 0 <= cx < T and 0 <= cy < T and x - x0 > -0.3 * w and x + w - x0 < T + 0.3 * w:
                lab.append(f"{c} {cx / T:.6f} {cy / T:.6f} {min(w, T) / T:.6f} {min(h, T) / T:.6f}")
        if lab: out.append((crop, lab))
    return out


if __name__ == '__main__':
    raw, ziel = sys.argv[1], sys.argv[2]; n_seiten = int(sys.argv[3]) if len(sys.argv) > 3 else 600
    rnd = random.Random(7); np.random.seed(7)
    sys_ = lade(raw); echt = lade_echt(sys.argv[4] if len(sys.argv) > 4 else None); print('Systeme', len(sys_), 'echte Seiten', len(echt))
    for teil in ('train', 'val'):
        os.makedirs(os.path.join(ziel, 'images', teil), exist_ok=True); os.makedirs(os.path.join(ziel, 'labels', teil), exist_ok=True)
    k = 0; jede = int(os.environ.get('ECHT_JEDE', '4'))   # jede n-te Seite ist eine echte (dritter Lernlauf: 2)
    for i in range(n_seiten):
        foto = False
        if echt and i % jede == jede - 1: pg, bx, foto = echte_seite(echt, rnd)
        else:
            pg, bx = seite(sys_, rnd)
            if rnd.random() < 0.2: pg = grob(pg, rnd)
        if foto: g = leicht(pg, rnd)
        else: g, bx = verschlechtern(pg, bx, rnd)
        teil = 'val' if i % 10 == 0 else 'train'
        for crop, lab in ausschnitte(g, bx, rnd, 3):
            fn = f"s{i:05d}_{k:06d}"; cv2.imwrite(os.path.join(ziel, 'images', teil, fn + '.jpg'), crop, [cv2.IMWRITE_JPEG_QUALITY, 92])
            open(os.path.join(ziel, 'labels', teil, fn + '.txt'), 'w').write('\n'.join(lab) + '\n'); k += 1
    with open(os.path.join(ziel, 'data.yaml'), 'w') as f:
        f.write(f"path: {os.path.abspath(ziel)}\ntrain: images/train\nval: images/val\nnames:\n" + ''.join(f"  {i}: {n}\n" for i, n in enumerate(KLASSEN)))
    print('Ausschnitte', k)


In [ ]:
%%writefile /content/Harmonikabau/griffschrift-wandler/omr-space/griffschrift-leser/yolo_fuchs_seiten.py
# Echte Lernseiten für den Kopf-Erkenner (YOLO) aus Fuchs-Blättern, deren Griffe aus den MIDI-Sätzen bekannt sind (pins.json):
# der bisherige Erkenner findet die Köpfe; Lernrahmen werden die Köpfe, deren Lage und Form zu den bekannten Griffen passt, und die
# übrigen, wenn der Erkenner sehr sicher ist (für die Form zählt die Tonhöhe nicht; nicht zugeordnet sind oft nur Wiederholungen,
# die der MIDI-Satz anders ausschreibt). Unsichere, nicht bestätigte Funde bleiben ohne Rahmen im Bild.
# Seiten, auf denen zu viele bekannte Griffe fehlen, werden ausgelassen. Nur jede zweite Seite (nach Namen) wird genommen,
# die übrigen bleiben für die Prüfung. Die Blätter selbst kommen nicht ins Repository.
# Aufruf: python3 yolo_fuchs_seiten.py FUCHSORDNER PINSORDNER AUSGABEORDNER [PRÜFLISTE]
#   FUCHSORDNER: <name>gs/GS*.gif bzw. u_<name>.png (hochskaliert), PINSORDNER: <name>.pins.json
import sys, os, glob, json
import numpy as np, cv2
HERE = os.path.dirname(os.path.abspath(__file__)); sys.path.insert(0, os.path.join(HERE, '..', 'gsleser'))
import gs2xml as X, detect as D
from align import chords_det, chords_pin, align, upscale

# Rahmengröße je Form in Linienabständen (Breite, Höhe), wie bei den vom Wandler gesetzten Lernbildern
GR = {0: (1.45, 1.15), 1: (1.45, 1.15), 2: (1.2, 1.2), 3: (1.35, 1.35), 4: (1.35, 1.1), 5: (1.35, 1.1)}


def klasse(h):
    if h['t'] == 'o': return 0 if h['fill'] else 1
    if h['t'] == 'x': return 2 if h['fill'] else 3
    return 4 if h['fill'] else 5


def seite(img, pins, mindest=0.9, sicher=0.9):
    g = X.load_gray(img)
    if g is None: return None
    work = '/tmp/_yfs.png'; cv2.imwrite(work, g)
    det = D.detect(work, 1); C = chords_det(det); B = chords_pin(pins)
    A = [{(h['p'], h['t']) for h in c} for c in C]; pr = align(A, B)
    gut = set(); tp = 0
    for i, j in pr:
        for h in C[i]:
            if (h['p'], h['t']) in B[j]: gut.add(id(h)); tp += 1
    nB = sum(len(b) for b in B)
    if nB == 0 or tp / nB < mindest: return dict(skip=True, recall=round(tp / max(1, nB), 3))
    boxen = []; weg = 0
    for c in C:
        for h in c:
            d = h['d']; k = klasse(h); w, hh = GR[k][0] * d, GR[k][1] * d
            if id(h) in gut or h.get('sc', 0) >= sicher: boxen.append((k, h['x'] - w / 2, h['y'] - hh / 2, w, hh))
            else: weg += 1
    return dict(g=g, boxen=boxen, recall=round(tp / nB, 3), weg=weg)


if __name__ == '__main__':
    fu, pn, ziel = sys.argv[1:4]; os.makedirs(ziel, exist_ok=True)
    namen = sorted(os.path.basename(f.rstrip('/'))[:-2] for f in glob.glob(os.path.join(fu, '*gs/')))
    # mit Prüfliste (viertes Argument, z. B. yolo_pruefseiten.txt): alle Seiten außer denen der Liste; sonst jede zweite wie bisher
    pruef = {l.strip() for l in open(sys.argv[4]) if l.strip()} if len(sys.argv) > 4 else None
    for n, k in enumerate(namen):
        if (k in pruef) if pruef is not None else n % 2: continue   # bleibt für die Prüfung
        if not os.path.exists(os.path.join(pn, k + '.pins.json')): continue
        img = os.path.join(os.path.dirname(fu.rstrip('/')), f'u_{k}.png')
        if not os.path.exists(img): upscale(glob.glob(os.path.join(fu, k + 'gs', 'GS*'))[0], img)
        r = seite(img, json.load(open(os.path.join(pn, k + '.pins.json')))['n'])
        if r is None or r.get('skip'): print(k, 'ausgelassen', r and r['recall']); continue
        cv2.imwrite(os.path.join(ziel, f'fuchs_{k}.png'), r['g'])
        json.dump(dict(boxes=[[int(c), round(x, 1), round(y, 1), round(w, 1), round(h, 1)] for c, x, y, w, h in r['boxen']]), open(os.path.join(ziel, f'fuchs_{k}.json'), 'w'))
        print(k, 'Köpfe', len(r['boxen']), 'ohne Rahmen', r['weg'], 'gefunden', r['recall'])


In [ ]:
%%writefile /content/Harmonikabau/griffschrift-wandler/omr-space/griffschrift-leser/yolo_vergleich_fotos.py
# Vergleich der Kopf-Erkenner auf Blättern ohne bekannte Lösung (Fotos, Handschrift, eingesandte PDFs): gezählt werden die Takte,
# die nicht aufgehen (Summe der Notenwerte ≠ Taktlänge), und die gelesenen Griffe, über den ganzen Leser (gs2xml.read).
# Ein Lauf je Erkenner, weil das gelernte Modell je Vorgang nur einmal geladen wird:
#   python3 yolo_vergleich_fotos.py alt BILD …
#   GS_YOLOMODELL=…/modell.onnx python3 yolo_vergleich_fotos.py yolo BILD …
# PDFs werden seitenweise als Bild gelesen (pymupdf, 200 dpi).
import sys, os, tempfile
HERE = os.path.dirname(os.path.abspath(__file__)); sys.path.insert(0, os.path.join(HERE, '..', 'gsleser'))
os.environ['GS_YOLOFOTO'] = '1'
import gs2xml as X


def seiten(p):
    if not p.lower().endswith('.pdf'): return [p]
    import fitz
    out = []
    for i, pg in enumerate(fitz.open(p)):
        f = os.path.join(tempfile.gettempdir(), f'_yvf_{os.getpid()}_{i}.png'); pg.get_pixmap(dpi=200).save(f); out.append(f)
    return out


if __name__ == '__main__':
    X.ERKENNER = sys.argv[1]; S = dict(takte=0, offen=0, griffe=0)
    for p in sys.argv[2:]:
        t = o = gr = 0
        for f in seiten(p):
            try: meas, L = X.read(f)
            except Exception as e: print(os.path.basename(p), 'Fehler', str(e)[:60]); continue
            for m, Lm in zip(meas, L):
                t += 1; o += sum(e['u'] for e in m) != Lm; gr += sum(1 for e in m if not e['rest'])
        print(f'{os.path.basename(p)}: Takte {t} offen {o} Griffe {gr}', flush=True)
        S['takte'] += t; S['offen'] += o; S['griffe'] += gr
    print('GESAMT', X.ERKENNER, os.environ.get('GS_YOLOMODELL', ''), S)


In [ ]:
#@title 2. Eingesandte Blätter und Fotos von Google Drive (griffschrift_daten)
# Quelle: Drive-Ordner griffschrift_daten; liegt der Ordner griffschriften noch im Repository, wird er zuerst dorthin kopiert
D = '/content/drive/MyDrive/griffschrift_daten'; R = '/content/Harmonikabau/griffschrift-wandler/griffschriften'
if os.path.isdir(R):
    !mkdir -p "$D" && cp -rn "$R"/. "$D"/
!ls "$D" | wc -l && du -sh "$D"
G = '/content/gs'   # ausgepackt zum Lernen und Prüfen
!mkdir -p $G/zip && cp -r "$D"/. $G/
!cd $G && for z in *.zip; do unzip -q -o "$z" -d "zip/${z%.zip}"; done; find $G/zip -type f | wc -l

In [ ]:
#@title 3. Zweites Modell von Drive holen (zum Vorbeschriften und zum Vergleich)
!cd $M && md5sum griffschrift_koepfe2.onnx
M2 = M + '/griffschrift_koepfe2.onnx'

In [ ]:
#@title 4. Fuchs-Blätter von volksmusik.cc holen (ca. 10 Minuten; nur erstes Blatt je Stück, nicht ins Repository)
import glob, re, time, urllib.request
PINS = '/content/Harmonikabau/griffschrift-wandler/midi/vm'
F = '/content/fuchs/fuchs'; os.makedirs(F, exist_ok=True)
namen = sorted(os.path.basename(p)[:-10] for p in glob.glob(PINS + '/*.pins.json'))
ok = 0
for k in namen:
    ziel = f'{F}/{k}gs'
    if glob.glob(ziel + '/GS*'): ok += 1; continue
    try:
        h = urllib.request.urlopen(f'https://www.volksmusik.cc/volkstanz/{k}gs.htm', timeout=20).read().decode('latin-1')
        m = re.search(r'src="images/(GS[^"]+?\.gif)"', h, re.I)
        if not m: print(k, 'kein Blatt'); continue
        os.makedirs(ziel, exist_ok=True)
        urllib.request.urlretrieve('https://www.volksmusik.cc/volkstanz/images/' + m.group(1), f'{ziel}/{m.group(1)}'); ok += 1
    except Exception as e: print(k, 'Fehler', str(e)[:60])
    time.sleep(0.5)
print('Blätter', ok, 'von', len(namen))
!cd /content/fuchs && zip -q -r -u $M/fuchs_seiten.zip fuchs

In [ ]:
#@title 5. Echte Lernseiten: Fuchs (alle außer Prüfseiten) und Fotos (ca. 30–40 Minuten)
%cd /content/Harmonikabau/griffschrift-wandler/omr-space/griffschrift-leser
!python3 yolo_fuchs_seiten.py /content/fuchs/fuchs/ $PINS /content/echt3 yolo_pruefseiten.txt 2>&1 | tail -n 5
!ls /content/echt3 | grep -c json
# Fotos zum Lernen: Muchitsch-Heft (Michael_Kara_*) und Rosenzopf Heft 2 (20261106_*); die Fotos vom 3.11. bleiben für die Prüfung
LERNFOTOS = sorted(glob.glob('/content/gs/Michael_Kara_*.jpg') + glob.glob('/content/gs/zip/GS-rosenzopf-2-2/**/*.jpg', recursive=True))
print('Lernfotos', len(LERNFOTOS))
import subprocess
r = subprocess.run(['python3', 'yolo_foto_seiten.py', '/content/echt3'] + LERNFOTOS, env=dict(os.environ, GS_YOLOMODELL=M2), capture_output=True, text=True)
print(r.stdout[-3000:], r.stderr[-1500:])
!ls /content/echt3/foto_*.json | wc -l
!cd /content && zip -q -r $M/echt3.zip echt3

In [ ]:
#@title 6. Trainingsbilder mit dem Wandler setzen (ca. 30–40 Minuten)
import subprocess, time
srv = subprocess.Popen(['python3','-m','http.server','8798'], cwd='/content/Harmonikabau/griffschrift-wandler', stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(2)
GEN = '/content/Harmonikabau/griffschrift-wandler/omr-space/griffschrift-leser/yolo_erzeuge_wandlerbilder.js'
for seed in (41, 53):
    !cd /content && NODE_PATH=/content/node_modules node {GEN} /content/raw3 {seed} 30
!ls /content/raw3 | wc -l

In [ ]:
#@title 7. Seiten zusammenstellen (jede zweite Seite eine echte, ca. 15 Minuten)
!cd /content/Harmonikabau/griffschrift-wandler/omr-space/griffschrift-leser && ECHT_JEDE=2 python3 yolo_daten.py /content/raw3 /content/ds3 2400 /content/echt3

In [ ]:
#@title 8. Lernen (ca. 2 Stunden auf einer T4; Gewichte laufend in Drive: griffschrift_modell/lauf3)
from ultralytics import YOLO
LAUF = M + '/lauf3'
letzt = LAUF + '/koepfe3/weights/last.pt'
if os.path.exists(letzt):   # nach einem Abbruch dort weitermachen
    YOLO(letzt).train(resume=True)
else:
    YOLO('yolov8s.pt').train(data='/content/ds3/data.yaml', imgsz=960, epochs=40, batch=16, patience=10, project=LAUF, name='koepfe3', exist_ok=True,
        degrees=0, fliplr=0, flipud=0, mosaic=0.5, hsv_h=0, hsv_s=0, hsv_v=0.3, scale=0.3, translate=0.05, close_mosaic=5)

In [ ]:
#@title 9. Als ONNX (halbe Genauigkeit) nach Drive
import shutil
YOLO(LAUF + '/koepfe3/weights/best.pt').export(format='onnx', imgsz=960, opset=12, simplify=True, half=True, device=0)
M3 = M + '/griffschrift_koepfe3.onnx'
shutil.copy(LAUF + '/koepfe3/weights/best.onnx', M3)
!cd $M && md5sum griffschrift_koepfe3.onnx | tee -a ergebnis3.txt && rm -f k3_teil_* && split -b 3m -d griffschrift_koepfe3.onnx k3_teil_ && ls -la k3_teil_*

In [ ]:
#@title 10. Prüfung an den 30 Fuchs-Prüfseiten: bisheriger Erkenner, zweites und drittes Modell (ca. 15 Minuten)
%cd /content/Harmonikabau/griffschrift-wandler/omr-space/griffschrift-leser
for name, mod in (('Modell 2', M2), ('Modell 3', M3)):
    r = subprocess.run(['python3', 'yolo_vergleich_fuchs.py', '/content/fuchs/fuchs'], env=dict(os.environ, GS_YOLOMODELL=mod), capture_output=True, text=True)
    log('== Fuchs-Prüfseiten,', name); log(r.stdout[-2500:]); print(r.stderr[-800:])

In [ ]:
#@title 11. Prüfung an nicht gelernten Fotos, Handschrift und PDFs: Takte offen, Griffe (ca. 45 Minuten)
%cd /content/Harmonikabau/griffschrift-wandler/omr-space/griffschrift-leser
fotos = sorted(set(glob.glob('/content/gs/zip/attachments*/**/*.jpg', recursive=True)), key=os.path.basename)
seen = set(); fotos = [f for f in fotos if not (os.path.basename(f) in seen or seen.add(os.path.basename(f)))] + ['/content/gs/20261103_165454.jpg']
pdfs = sorted(f for f in glob.glob('/content/gs/*.pdf') + glob.glob('/content/gs/*.PDF') if 'Bassbelegung' not in f)
einsendungen = sorted(f for f in glob.glob('/content/gs/zip/*/**/*.pdf', recursive=True) if '/attachments' not in f)
for gruppe, liste in (('Fotos 3.11.', fotos), ('PDFs', pdfs), ('eingesandte Stücke', einsendungen)):
    for name, mod, erk in (('bisher', '', 'alt'), ('Modell 2', M2, 'yolo'), ('Modell 3', M3, 'yolo')):
        r = subprocess.run(['python3', 'yolo_vergleich_fotos.py', erk] + liste, env=dict(os.environ, GS_YOLOMODELL=mod), capture_output=True, text=True)
        log('==', gruppe, name); log(r.stdout[-4000:]); print(r.stderr[-500:])